# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmerSajid842/flyrankmlproject/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load dataset
df = pd.read_csv("content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

# Select features
feature_columns = [
    "search_volume",
    "competition",
    "word_count",
    "content_age_days",
    "ctr"
]

# Create feature dataframe
feature_df = df[feature_columns].copy()

feature_df.head()

Dataset Shape: (30000, 44)


,search_volume,competition,word_count,content_age_days,ctr
0,10.0,0.67,3221.0,187,0.76
1,90.0,0.01,2481.0,445,0.05
2,0.0,0.00,3515.0,141,0.09
3,10.0,0.00,NaN,463,0.49
4,0.0,0.00,2803.0,263,0.13


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_notes = pd.DataFrame({
    "Feature": feature_columns,
    "Meaning": [
        "Monthly keyword search volume",
        "Keyword competition score",
        "Number of words in content",
        "Age of content in days",
        "Click-through rate"
    ],
    "Missing Values": [
        df["search_volume"].isnull().sum(),
        df["competition"].isnull().sum(),
        df["word_count"].isnull().sum(),
        df["content_age_days"].isnull().sum(),
        df["ctr"].isnull().sum()
    ],
    "Categorical": [
        "No",
        "No",
        "No",
        "No",
        "No"
    ],
    "Available at Decision Time": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

feature_notes


,Feature,Meaning,Missing Values,Categorical,Available at Decision Time
0,search_volume,Monthly keyword search volume,2468,No,Yes
1,competition,Keyword competition score,2468,No,Yes
2,word_count,Number of words in content,7699,No,Yes
3,content_age_days,Age of content in days,0,No,Yes
4,ctr,Click-through rate,0,No,Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Honest features
X = df[feature_columns].fillna(0)
y = df["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

honest_accuracy = accuracy_score(y_test, pred)

print("Honest Accuracy:", honest_accuracy)

# Leakage feature
X_leak = df[feature_columns + ["trend_direction"]]

X_leak = pd.get_dummies(X_leak)

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

leak_accuracy = accuracy_score(y_test, pred)

print("Leakage Accuracy:", leak_accuracy)

Honest Accuracy: 0.6231666666666666
Leakage Accuracy: 1.0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("""
Excluded Features

1. trend_direction
Reason:
This column directly defines the target label. Using it would leak the answer into the model.

2. trend_pct
Reason:
It represents future performance and would not be known at prediction time.

3. content_id
Reason:
Unique identifier with no predictive value.

4. client_id
Reason:
Identifier only, not useful for generalization.
""")


Excluded Features

1. trend_direction
Reason:
This column directly defines the target label. Using it would leak the answer into the model.

2. trend_pct
Reason:
It represents future performance and would not be known at prediction time.

3. content_id
Reason:
Unique identifier with no predictive value.

4. client_id
Reason:
Identifier only, not useful for generalization.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.